In [195]:
from bs4 import BeautifulSoup
import time
import pandas as pd
import re

In [196]:
soup_list = []

for i in range(22):
    with open(f"oab_questions_filtered/oab_questions{i+1}.html", "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
        soup_list.append(soup)

In [197]:
def get_exams_name(soup_list):
    exams = []
    years = []
    for soup in soup_list:
        for prova in soup.find_all("div", class_="q-question-info"):
            text = re.sub(r'\s+', ' ', prova.get_text()).strip()
            ano = text.split("Banca")[0].replace("Ano:", "")
            years.append(ano)
            exams.append(text.split("Prova:")[1].replace("|",""))
    return years, exams

In [198]:
years, provas = get_exams_name(soup_list)

In [199]:
def get_questions(soup_list):
    questions = []
    for soup in soup_list:
        for question in soup.find_all("div", class_="q-question-enunciation"):
            questions.append(re.sub(r'\s+', ' ', question.get_text()).strip())
    return questions

In [200]:
questions = get_questions(soup_list)

In [201]:
def get_options(soup_list):
    options = []
    for soup in soup_list:
        for option in soup.find_all("div", class_="q-item-enum js-alternative-content"):
            options.append(re.sub(r'\s+', ' ', option.get_text()).strip())
    return options

In [202]:
options = get_options(soup_list)

In [203]:
questions_json = []
for i in range(len(questions)):
    dict_question = {"question_number": i+1, "question": questions[i], "options": options[i*4:(i+1)*4], "exam": provas[i], "answer":""}  # Assuming each question has 4 options
    questions_json.append(dict_question)

In [204]:
df_questions = pd.DataFrame(questions_json)

In [205]:
df_questions_filtered = df_questions[df_questions['exam'].str.contains('Unificado')] 

In [206]:
df_questions_filtered['question_number'] = range(1, len(df_questions_filtered) + 1)

In [207]:
def fix_space_punctuation(texto):
    return re.sub(r'([.!?,;:])([A-ZÀ-Ú])', r'\1 \2', texto)

def fix_options(options):
    fixed_options = []
    for option in options:
        fixed_option = option.replace('\n', ' ')
        fixed_options.append(fixed_option)
    return fixed_options

In [208]:
df_questions_filtered['question'] = df_questions_filtered['question'].str.replace('\n',' ')
df_questions_filtered['options'] = df_questions_filtered['options'].apply(fix_options)
df_questions_filtered['question'] = df_questions_filtered['question'].apply(fix_space_punctuation)
df_questions_filtered['question'] = df_questions_filtered['question'].str.replace('  ', ' ')

In [209]:
df_questions_filtered.to_csv('csvs/oab/oab_questions_unified_no_answer.csv', index=False, encoding='utf-8-sig')